## FewShotPromptTemplate과 시스템 메시지 활용 

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

In [ ]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0)

In [ ]:
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    {
        "news": "손흥민이 프리미어리그 경기에서 시즌 10호 골을 기록하며 팀의 3-1 승리를 이끌었다. 이로써 그는 프리미어리그 진출 이후 9시즌 연속 두 자릿수 득점이라는 대기록을 세웠다.",
        "keywords": "손흥민, 프리미어리그, 두자릿수득점",
    },
    {
        "news": "한국은행 금융통화위원회는 물가 상승세가 둔화됨에 따라 기준금리를 0.25%포인트 인하한다고 발표했다. 시장에서는 이번 결정이 가계 대출 부담 완화와 내수 회복에 도움이 될 것으로 기대하고 있다.",
        "keywords": "한국은행, 기준금리, 금리인하",
    },
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 뉴스 키워드 추출 전문가입니다.\n"
     "뉴스에서 가장 핵심적인 키워드 정확히 3개를 추출하세요.\n"
     "규칙:\n"
     "- 반드시 '키워드: A, B, C' 형식 한 줄로만 답하세요.\n"
     "- 고유명사(기업, 기관, 인물, 제품)를 우선하고, 복합 명사는 띄어쓰기 없이 붙여 쓰세요.\n"
     "- 설명이나 다른 문장은 출력하지 마세요."),
    few_shot_prompt,
    ("human", "{input}"),
])

for msg in final_prompt.invoke({"input": "(테스트)"}).to_messages():
    msg.pretty_print()

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = final_prompt | llm | StrOutputParser()

test_news = "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다."

print(chain.invoke({"input": test_news}))

In [ ]:
test_cases = {
    "IT":   test_news,
    "환경": "환경부는 올여름 폭염과 집중호우가 잇따르자 기후위기 적응 대책을 강화하겠다고 밝혔다. 특히 도심 홍수 예방을 위한 빗물 저류시설을 2030년까지 두 배로 늘릴 계획이다.",
    "문화": "방탄소년단(BTS)이 완전체로 복귀해 새 앨범을 발표했다. 앨범은 발매 첫날 미국 빌보드 차트 1위에 오르며 K팝의 글로벌 영향력을 다시 한번 입증했다.",
    "사회": "교육부는 2028학년도 대입 개편안을 확정하고, 수능 선택과목을 폐지해 모든 수험생이 같은 과목으로 시험을 치르도록 했다.",
}

results = chain.batch([{"input": n} for n in test_cases.values()])

for field, res in zip(test_cases, results):
    # 형식 검증: "키워드:"로 시작하고, 쉼표로 나눈 키워드가 정확히 3개인지
    kws = [k.strip() for k in res.replace("키워드:", "").split(",")]
    ok = res.startswith("키워드:") and len(kws) == 3
    print(f"[{field}] {'' if ok else '형식 불일치'}  {res}")

In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

str_example_prompt = PromptTemplate.from_template("뉴스: {news}\n키워드: {keywords}")

str_few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=str_example_prompt,
    prefix="당신은 뉴스 키워드 추출 전문가입니다. 아래 예시처럼 핵심 키워드 3개를 추출하세요.",
    suffix="뉴스: {input}\n키워드:",
    input_variables=["input"],
)

print(str_few_shot.format(input="(테스트)"))

str_chain = str_few_shot | llm | StrOutputParser()
print("키워드:", str_chain.invoke({"input": test_news}).strip())